In [0]:
df = spark.table("gizmo_box.bronze.v_customer")
display(df)

## Remove null customer_id

In [0]:
df = df.filter(df.customer_id.isNotNull())
#df = spark.sql("select * from gizmo_box.bronze.v_customer where customer_id is not null")
display(df)

## Remove exact duplicate event

In [0]:
df = df.dropDuplicates()

df = spark.sql("select distinct * from gizmo_box.bronze.v_customer where customer_id is not null")
df.createOrReplaceTempView("distinct_customer")
display(df)

## Remove duplicate customer_id using created timestamp

In [0]:
from pyspark.sql import functions as F


df_group = df.groupBy("customer_id").agg(F.max("created_timestamp").alias("max_created_timestamp"))

join_df = df.join(df_group, ((df["customer_id"] == df_group["customer_id"]) & (df["created_timestamp"] == df_group["max_created_timestamp"])), "inner").select(df["*"])

display(join_df)

## CAST data to proper format

In [0]:
from pyspark.sql.functions import col

# date formate should be in yyyy-MM-dd 
formateddf= join_df.select(join_df.customer_id, join_df.customer_name, join_df.email, join_df.telephone, col("created_timestamp").cast("timestamp"), col("date_of_birth").cast("date"), col("member_since").cast("date"), join_df.filepath)

formateddf.createOrReplaceTempView("customer_formatted_data")

display(formateddf)

In [0]:
%sql
CREATE TABLE gizmo_box.silver.customer as
select * from customer_formatted_data;